# Background

In [14]:
import numpy as np
from src.custom_fastkan import FastKAN, FastKANLayer
import gurobipy as gp
from gurobipy import GRB
import torch
from torch import nn

def fit_line_through_points(x1, y1, x2, y2):
    slope = (y2 - y1) / (x2 - x1)
    intercept = y1 - slope * x1
    return slope, intercept

def find_all_bspline_segments(layer, input_index, output_index, max_segments):
    n_samples = 25
    # Fetch curve data ONCE
    x_tensor, y_tensor = layer.plot_curve(input_index, output_index, num_pts=n_samples)
    x_points, y_points = x_tensor.detach().cpu().numpy(), y_tensor.detach().cpu().numpy()
    
    # 1. Precompute errors for all possible segments (ONCE)
    errors = np.full((n_samples, n_samples), np.inf)
    for start_idx in range(n_samples-1):
        x_start = x_points[start_idx]
        y_start = y_points[start_idx]
        for end_idx in range(start_idx + 1, n_samples):
            x_end = x_points[end_idx]
            y_end = y_points[end_idx]
            slope, intercept = fit_line_through_points(x_start, y_start, x_end, y_end)
            segment_x = x_points[start_idx:end_idx+1]
            segment_y = y_points[start_idx:end_idx+1]
            predicted_y = slope * segment_x + intercept
            segment_error = np.max(np.abs(predicted_y - segment_y))
            errors[start_idx, end_idx] = segment_error
    
    # 2. Dynamic programming to find optimal segmentation (ONCE)
    dp_table = np.full((max_segments, n_samples), np.inf)
    backtrack = np.zeros((max_segments, n_samples), dtype=int)
    
    # Base Case: 1 segment
    for j in range(1, n_samples):
        dp_table[0, j] = errors[0, j]
    
    # Recurrence
    for i in range(1, max_segments):
        for j in range(i+1, n_samples):
            # We try breaking at k, where k is between the start of this segment number and current point
            # k is the END of the previous segment
            for k in range(i, j): 
                if dp_table[i-1, k] == np.inf: continue
                if errors[k, j] == np.inf: continue
                
                curr_error = max(dp_table[i-1, k], errors[k, j])
                if curr_error < dp_table[i, j]:
                    dp_table[i, j] = curr_error
                    backtrack[i, j] = k

    # 3. Extract results for EVERY segment count from 1 to max_segments
    results = {}
    
    for seg_count in range(1, max_segments + 1):
        row_idx = seg_count - 1
        final_error = dp_table[row_idx, n_samples-1]
        
        # If optimization failed for this count, skip or return infinity
        if final_error == np.inf:
            results[seg_count] = ([], np.inf)
            continue

        # Reconstruct segments for this specific count using the backtrack table
        segments = []
        curr_seg_end = n_samples - 1
        
        for i in range(row_idx, -1, -1):
            if i > 0:
                prev_seg_end = backtrack[i, curr_seg_end]
            else:
                prev_seg_end = 0
            
            x1, y1 = x_points[prev_seg_end], y_points[prev_seg_end]
            x2, y2 = x_points[curr_seg_end], y_points[curr_seg_end]
            slope, intercept = fit_line_through_points(x1, y1, x2, y2)
            segments.insert(0, (x1, x2, slope, intercept))
            
            curr_seg_end = prev_seg_end
        
        results[seg_count] = (segments, final_error)
        
    return results

def calculate_bspline_lipschitz_constant(layer: 'FastKANLayer', input_idx: int, output_idx: int, num_pts: int = 1000, min_x: float = None, max_x: float = None) -> float:
    if min_x is None:
        min_x = layer.rbf.grid_min
    if max_x is None:
        max_x = layer.rbf.grid_max
    x_tensor, y_tensor = layer.plot_curve(input_idx, output_idx, num_pts=num_pts)
    x_np = x_tensor.detach().cpu().numpy()
    y_np = y_tensor.detach().cpu().numpy()
    mask = (x_np >= min_x) & (x_np <= max_x)
    x_np = x_np[mask]
    y_np = y_np[mask]

    dx = np.diff(x_np)
    dy = np.diff(y_np)
    nonzero_dx = dx != 0
    slopes = np.zeros_like(dx)
    slopes[nonzero_dx] = dy[nonzero_dx] / dx[nonzero_dx]
    lipschitz_constant = np.max(np.abs(slopes))
    return lipschitz_constant

def compute_dp_tables_lipschitz(kan_model, max_segments=15):
    error_tables = {}
    segments_tables = {}
    lipschitz_constants = {}
    
    for layer_idx, layer in enumerate(kan_model.layers):
        input_dim = layer.input_dim
        output_dim = layer.output_dim
        print(f"Analyzing Layer {layer_idx}: {input_dim} inputs -> {output_dim} outputs")
        
        for input_idx in range(input_dim):
            for output_idx in range(output_dim):
                spline_key = (layer_idx, input_idx, output_idx)
                all_results = find_all_bspline_segments(layer, input_idx, output_index=output_idx, max_segments=max_segments)
                
                error_table = {}
                segments_table = {}
                
                for k, (segs, err) in all_results.items():
                    segments_table[k] = segs
                    error_table[k] = err
                lipschitz_constant = calculate_bspline_lipschitz_constant(layer=layer, input_idx=input_idx, output_idx=output_idx)
                
                error_tables[spline_key] = error_table
                segments_tables[spline_key] = segments_table
                lipschitz_constants[spline_key] = lipschitz_constant
    
    return error_tables, segments_tables, lipschitz_constants


def weight_dp_tables_lipschitz(kan_model, error_tables, segments_tables, lipschitz_constants):
    node_sensitivities = {}
    num_layers = len(kan_model.layers)
    
    # Intialize final layer with all 1's
    final_layer = kan_model.layers[num_layers - 1]
    for out_idx in range(final_layer.output_dim):
        node_sensitivities[(num_layers, out_idx)] = 1.0

    # Calculate sensitivity for the inputs of a layer based on the output layer's sensitivities
    layer_idx = num_layers - 1
    while layer_idx >= 0:
        current_layer = kan_model.layers[layer_idx]
        for input_idx in range(current_layer.input_dim):
            sensitivity_sum = 0.0
            for output_idx in range(current_layer.output_dim):
                lipschitz_constant = lipschitz_constants.get((layer_idx, input_idx, output_idx), 0.0)
                child_node_sensitivity = node_sensitivities[(layer_idx + 1, output_idx)]
                sensitivity_sum += lipschitz_constant * child_node_sensitivity
            node_sensitivities[(layer_idx, input_idx)] = sensitivity_sum
        layer_idx -= 1

    # Weight the error table (just do sensitivities * table)
    weighted_error_tables = {}
    for layer_idx, layer in enumerate(kan_model.layers):
        for input_idx in range(layer.input_dim):
            for output_idx in range(layer.output_dim):
                current_bspline_key = (layer_idx, input_idx, output_idx)
                # The error of a spline is added directly to its destination node (output_idx)
                sensitivity_of_current_output_node = node_sensitivities[(layer_idx + 1, output_idx)]
                weighted_table_for_bspline = {}
                for pair in error_tables[current_bspline_key].items():
                    num_segments = pair[0]
                    error = pair[1]
                    weighted_table_for_bspline[num_segments] = error * sensitivity_of_current_output_node
                weighted_error_tables[current_bspline_key] = weighted_table_for_bspline
    return weighted_error_tables


def solve_best_segment_allocation(weighted_error_tables, target_max_error):
    model = gp.Model()
    # Create binary variable x[spline_key, num_segments] for every choice and every spline
    x = {}
    binary_vars_for_splines = {} 
    for spline_key, num_segment_options in weighted_error_tables.items():
        binary_vars_for_splines[spline_key] = []
        for pair in num_segment_options.items():
            k_segments = pair[0]
            error_val = pair[1]
            if np.isinf(error_val):
                continue
            x[(spline_key, k_segments)] = model.addVar(vtype=GRB.BINARY, obj=k_segments)
            binary_vars_for_splines[spline_key].append(x[(spline_key, k_segments)])
    model.update()

    # Make sure we can only pick 1 variable
    for pair in binary_vars_for_splines.items():
        spline_key = pair[0]
        variables = pair[1]
        model.addConstr(gp.quicksum(variables) == 1)
    # Make sure we're under the max error
    total_error_expr = gp.quicksum(weighted_error_tables[s_key][k] * x[(s_key, k)] for s_key, k in x.keys())
    model.addConstr(total_error_expr <= target_max_error)

    # Minimize the total number of segments
    total_segments_expr = gp.quicksum(
        k * x[(s_key, k)] 
        for s_key, k in x.keys()
    )
    model.setObjective(total_segments_expr, GRB.MINIMIZE)

    # Optimize!
    model.optimize()
    optimal_allocation = {}
    if model.status == GRB.OPTIMAL:
        print(f"Optimization successful! Minimum Segments Required: {model.objVal}")
        for (spline_key, k_segments), variable in x.items():
            if variable.X > 0.5:
                optimal_allocation[spline_key] = k_segments
        return optimal_allocation, model.objVal
    elif model.status == GRB.INFEASIBLE:
        print("Model is INFEASIBLE. It is impossible to achieve this low of an error with the available segment options.")
        return None, float('inf')
    else:
        print(f"Optimization ended with status code: {model.status}")
        return None, float('inf')

def build_kan_milp_model(kan_shape, segments_tables, error_tables, optimal_allocation, x_min_vec, x_max_vec):
    model = gp.Model()
    input_dim = kan_shape[0] 
    all_layer_variables = []

    # Create Input Variables
    current_layer_range_variables = []
    for i in range(input_dim):
        input_range_variable = model.addVar(lb=x_min_vec[i], ub=x_max_vec[i])
        current_layer_range_variables.append(input_range_variable)
    all_layer_variables.append(current_layer_range_variables)

    # Build Hidden Layers Sequentially
    num_transitions = len(kan_shape) - 1 # If shape is [784, 32, 10], we have 2 transitions: 0->1 and 1->2
    for layer_idx in range(num_transitions):
        layer_input_dimension = kan_shape[layer_idx]
        layer_output_dimension = kan_shape[layer_idx + 1]
        
        # Initialize output variables for this layer (by creating our list of inputs to the next layer)
        next_layer_input_ranges = []
        for i in range(layer_output_dimension): 
            next_layer_input_ranges.append(gp.LinExpr()) # ex. layer2_neuron_i = layer1_neuron1 + layer1_neuron2 + ... (linear expression!)
        
        for src_idx in range(layer_input_dimension):
            for dst_idx in range(layer_output_dimension):
                # Extract PWL Points
                num_segs = optimal_allocation[(layer_idx, src_idx, dst_idx)]
                seg_data = segments_tables[(layer_idx, src_idx, dst_idx)][num_segs] # table has all possible segment allocations, only extract the optimal one (num_segs)
                x_pts, y_pts = [], []
                for idx, (x1, x2, slope, intercept) in enumerate(seg_data):
                    x_pts.append(x1)
                    y_pts.append(slope * x1 + intercept)
                    if idx == len(seg_data) - 1: # Add end point of last segment
                        x_pts.append(x2)
                        y_pts.append(slope * x2 + intercept)

                # Create output variable for each bspline (adding in error)
                approx_error = error_tables[(layer_idx, src_idx, dst_idx)][num_segs]
                error_var = model.addVar(lb=-approx_error, ub=approx_error)
                src_var = current_layer_range_variables[src_idx]
                result_var = model.addVar(lb=-GRB.INFINITY, ub=GRB.INFINITY)
                model.addGenConstrPWL(src_var, result_var, x_pts, y_pts) # "The relationship between src_var and result_var must follow the path connected by the dots"
                next_layer_input_ranges[dst_idx] += result_var + error_var # fill in the output linear expressions we defined earlier (for each neuron in the next layer)

        # Create variables for next layer nodes
        next_layer_range_variables = []
        for j in range(layer_output_dimension):
            next_layer_range_variable = model.addVar(lb=-GRB.INFINITY, ub=GRB.INFINITY)
            model.addConstr(next_layer_range_variable == next_layer_input_ranges[j])
            next_layer_range_variables.append(next_layer_range_variable)
        current_layer_range_variables = next_layer_range_variables
        
        all_layer_variables.append(current_layer_range_variables)

    model.update()
    return model, all_layer_variables

def solve_kan_interval_milp(kan_shape, segments_tables, error_tables, optimal_allocation, hidden_layer_mip_gap, output_layer_mip_gap, x_min_vec, x_max_vec):
    model, all_layer_variables = build_kan_milp_model(
        kan_shape, 
        segments_tables, 
        error_tables,
        optimal_allocation, 
        x_min_vec, 
        x_max_vec
    )
    
    # Iterate through layers. We start from index 1 because index 0 is the input (already bounded)
    for layer_idx in range(1, len(all_layer_variables)):
        current_layer_vars = all_layer_variables[layer_idx]
        print(f"Optimizing bounds for Layer {layer_idx} ({len(current_layer_vars)} neurons)...")
        if layer_idx < len(all_layer_variables) - 1:
            model.setParam('MIPGap', hidden_layer_mip_gap) 
            model.setParam('TimeLimit', 5.0)
        else:
            model.setParam('MIPGap', output_layer_mip_gap)
        for i, var in enumerate(current_layer_vars):
            # Update each variable's lower bound and upper bound in the model (restricts the search space for NEXT layer significantly)
            model.setObjective(var, GRB.MINIMIZE)
            model.optimize()
            if model.status == GRB.OPTIMAL:
                var.lb = model.ObjBound
            else:
                var.lb = -np.inf
            model.setObjective(var, GRB.MAXIMIZE)
            model.optimize()
            if model.status == GRB.OPTIMAL:
                var.ub = model.ObjBound
            else:
                var.ub = np.inf   
        model.update() # Apply the bound updates to the model so the next layer sees them

    # The result we want is the bounds of the very last layer
    final_layer_vars = all_layer_variables[-1]
    min_bounds = np.zeros(len(final_layer_vars))
    max_bounds = np.zeros(len(final_layer_vars))
    for i, var in enumerate(final_layer_vars):
        min_bounds[i] = var.lb
        max_bounds[i] = var.ub
    return min_bounds, max_bounds

# Gurobi Verification Comparison

In [ ]:
def get_spline_bounds(segments, x_min, x_max):
    y_min, y_max = np.inf, -np.inf
    for (sx1, sx2, slope, intercept) in segments:
        # Find the intersection of the segment domain and input domain
        overlap_start = max(sx1, x_min)
        overlap_end = min(sx2, x_max)
        # If they overlap, evaluate the line at the edges
        if overlap_start <= overlap_end:
            val_start = slope * overlap_start + intercept
            val_end = slope * overlap_end + intercept
            y_min = min(y_min, val_start, val_end)
            y_max = max(y_max, val_start, val_end)
    return y_min, y_max


def propagate_kan_intervals(kan_shape, segments_tables, optimal_allocation, input_lb, input_ub):
    layer_bounds = []
    # Input layer
    current_lb = input_lb
    current_ub = input_ub
    layer_bounds.append((current_lb, current_ub))
    num_transitions = len(kan_shape) - 1

    # Propagate across all layers 
    for layer_idx in range(num_transitions):
        in_dim = kan_shape[layer_idx]
        out_dim = kan_shape[layer_idx + 1]
        next_lb = np.zeros(out_dim)
        next_ub = np.zeros(out_dim)
        for dst in range(out_dim):
            # Min_sum = Sum(Min_parts), Max_sum = Sum(Max_parts)
            total_min = 0.0
            total_max = 0.0
            for src in range(in_dim):
                x_min = current_lb[src]
                x_max = current_ub[src]
                num_segs = optimal_allocation[(layer_idx, src, dst)]
                segs = segments_tables[(layer_idx, src, dst)][num_segs]

                s_min, s_max = get_spline_bounds(segs, x_min, x_max)
                total_min += s_min
                total_max += s_max
            next_lb[dst] = total_min
            next_ub[dst] = total_max
        layer_bounds.append((next_lb, next_ub))
        current_lb, current_ub = next_lb, next_ub
    return layer_bounds

def solve_kan_interval_milp(kan_shape, segments_tables, error_tables, optimal_allocation, output_layer_mip_gap, x_min_vec, x_max_vec):
    precomputed_bounds = propagate_kan_intervals(
        kan_shape, 
        segments_tables, 
        optimal_allocation, 
        x_min_vec, 
        x_max_vec
    )
    model, all_layer_variables = build_kan_milp_model(
        kan_shape, 
        segments_tables, 
        error_tables,
        optimal_allocation, 
        x_min_vec, 
        x_max_vec
    )

    # Apply the precomputed bounds
    for layer_idx, vars_in_layer in enumerate(all_layer_variables):
        lbs, ubs = precomputed_bounds[layer_idx]
        for neuron_idx, var in enumerate(vars_in_layer):
            var.lb = lbs[neuron_idx]
            var.ub = ubs[neuron_idx]
            
    model.update()
    # Solve only the final layer
    final_layer_vars = all_layer_variables[-1]
    min_bounds = np.zeros(len(final_layer_vars))
    max_bounds = np.zeros(len(final_layer_vars))
    print(f"Starting MILP verification for {len(final_layer_vars)} output neurons...")
    model.setParam('MIPGap', output_layer_mip_gap)
    for i, var in enumerate(final_layer_vars):
        print(f"  Optimizing Output {i}...")
        # Minimize
        model.setObjective(var, GRB.MINIMIZE)
        model.optimize()
        min_bounds[i] = model.ObjBound
        # Maximize
        model.setObjective(var, GRB.MAXIMIZE)
        model.optimize()
        max_bounds[i] = model.ObjBound
    return min_bounds, max_bounds

In [21]:
my_kan = FastKAN([10, 5, 5], num_grids=8, use_base_update=False, use_layernorm=False)
error_tables, segments_tables, lipschitz_constants = compute_dp_tables_lipschitz(my_kan, 30)
weighted_error_tables = weight_dp_tables_lipschitz(my_kan, error_tables, segments_tables, lipschitz_constants)
optimal_allocation, min_error = solve_best_segment_allocation(weighted_error_tables, 100.0)
kan_shape = [my_kan.layers[0].input_dim]
for layer in my_kan.layers:
    kan_shape.append(layer.output_dim)
x_min_vec = np.full(kan_shape[0], -1.0)
x_max_vec = np.full(kan_shape[0], 1.0)

hidden_layer_mip_gap = 0.1
output_layer_mip_gap = 0.1 #1e-4
min_outputs, max_outputs = solve_kan_interval_milp(
    kan_shape,
    segments_tables, 
    error_tables,
    optimal_allocation, 
    #hidden_layer_mip_gap,
    output_layer_mip_gap,
    x_min_vec, 
    x_max_vec
)

print("Minimum Output Bounds:", min_outputs)
print("Maximum Output Bounds:", max_outputs)

Analyzing Layer 0: 10 inputs -> 5 outputs
Analyzing Layer 1: 5 inputs -> 5 outputs
Gurobi Optimizer version 12.0.1 build v12.0.1rc0 (mac64[arm] - Darwin 24.6.0 24G90)

CPU model: Apple M2 Pro
Thread count: 10 physical cores, 10 logical processors, using up to 10 threads

Optimize a model with 76 rows, 1800 columns and 3600 nonzeros
Model fingerprint: 0x61082fb7
Variable types: 0 continuous, 1800 integer (1800 binary)
Coefficient statistics:
  Matrix range     [9e-09, 1e+00]
  Objective range  [1e+00, 2e+01]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 1e+02]
Found heuristic solution: objective 927.0000000
Presolve removed 76 rows and 1800 columns
Presolve time: 0.00s
Presolve: All rows and columns removed

Explored 0 nodes (0 simplex iterations) in 0.01 seconds (0.00 work units)
Thread count was 1 (of 10 available processors)

Solution count 2: 75 927 

Optimal solution found (tolerance 1.00e-04)
Best objective 7.500000000000e+01, best bound 7.500000000000e+01, gap 0.00

In [19]:
import gurobipy as gp
from gurobipy import GRB
import torch
import torch.nn as nn
import numpy as np

# --- 1. The Full Network Verification Function ---
def verify_full_network(model, input_lb, input_ub, timeout=300):
    """
    Verifies a simple MLP (Linear -> ReLU -> Linear...).
    Calculates the MIN and MAX for EVERY output neuron.
    """
    print("\n--- Building Gurobi Model for Full Network ---")
    m = gp.Model("mlp_verification")
    m.setParam("TimeLimit", timeout)
    # m.setParam("OutputFlag", 0)  # Silence Gurobi logs if desired

    # A. Create Input Variables
    input_vars = []
    print(f"Creating {len(input_lb)} input variables...")
    for i in range(len(input_lb)):
        # Define inputs with the global lower/upper bounds
        v = m.addVar(lb=input_lb[i], ub=input_ub[i], name=f"inp_{i}")
        input_vars.append(v)
    
    current_vars = input_vars
    
    # B. Build the constraint graph (Linear -> ReLU layers)
    layer_idx = 0
    for layer in model.net:
        if isinstance(layer, torch.nn.Flatten):
            continue
            
        elif isinstance(layer, torch.nn.Linear):
            weight = layer.weight.detach().cpu().numpy()
            bias = layer.bias.detach().cpu().numpy()
            out_dim, in_dim = weight.shape
            
            print(f"  > Adding Linear Layer {layer_idx}: {in_dim} inputs -> {out_dim} outputs")
            next_vars = []
            for j in range(out_dim):
                lin_expr = gp.LinExpr(weight[j, :], current_vars) + bias[j]
                # Unbounded intermediate variable
                out_v = m.addVar(lb=-GRB.INFINITY, ub=GRB.INFINITY, name=f"lay{layer_idx}_{j}")
                m.addConstr(out_v == lin_expr)
                next_vars.append(out_v)
            
            current_vars = next_vars
            layer_idx += 1

        elif isinstance(layer, torch.nn.ReLU):
            print(f"  > Adding ReLU constraints...")
            next_vars = []
            for j, pre_var in enumerate(current_vars):
                # Big-M bounds. CAUTION: If your network weights are huge, increase these.
                l_bound, u_bound = -100.0, 100.0 
                
                post_var = m.addVar(lb=0.0, ub=u_bound, name=f"relu{layer_idx}_{j}")
                z = m.addVar(vtype=GRB.BINARY, name=f"relu_state{layer_idx}_{j}")
                
                # y = max(0, x) mixed-integer constraints
                m.addConstr(post_var >= 0)
                m.addConstr(post_var >= pre_var)
                m.addConstr(post_var <= u_bound * z)
                m.addConstr(post_var <= pre_var - l_bound * (1 - z))
                
                next_vars.append(post_var)
            current_vars = next_vars

    # The final variables in 'current_vars' are the network outputs
    output_vars = current_vars
    num_outputs = len(output_vars)
    results = []

    print(f"\n--- Starting Optimization for {num_outputs} Output Neurons ---")
    
    # C. Loop through every output neuron
    for i in range(num_outputs):
        target_var = output_vars[i]
        print(f"Processing Output Node {i}...")

        # 1. Find Maximum
        m.setObjective(target_var, GRB.MAXIMIZE)
        m.optimize()
        
        if m.status == GRB.OPTIMAL:
            val_max = m.objVal
        elif m.status == GRB.TIME_LIMIT:
            val_max = m.ObjBound
        else:
            val_max = float('nan')

        # 2. Find Minimum
        m.setObjective(target_var, GRB.MINIMIZE)
        m.optimize()

        if m.status == GRB.OPTIMAL:
            val_min = m.objVal
        elif m.status == GRB.TIME_LIMIT:
            val_min = m.ObjBound
        else:
            val_min = float('nan')

        results.append((val_min, val_max))
        print(f"  Node {i}: range [{val_min:.4f}, {val_max:.4f}]")

    return results

# --- 2. Setup Model ---
class TinyMLP(nn.Module):
    def __init__(self):
        super(TinyMLP, self).__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(28, 16),
            nn.ReLU(),
            nn.Linear(16, 10) # 10 Output classes
        )

# Initialize
mlp_model = TinyMLP()

# --- 3. Define Global Bounds ---
input_dim = 28
# Range [-1.0, 1.0] for every input
x_min_vec = np.full(input_dim, -1.0) 
x_max_vec = np.full(input_dim, 1.0)

# --- 4. Run Verification ---
bounds_results = verify_full_network(
    model=mlp_model, 
    input_lb=x_min_vec, 
    input_ub=x_max_vec,
    timeout=30 # 30s per solve (total time = 30s * 2 * 10 outputs)
)

# --- 5. Pretty Print Final Report ---
print("\n" + "="*40)
print(f"{'Neuron Idx':<12} | {'Min Val':<12} | {'Max Val':<12}")
print("-" * 40)
for idx, (min_v, max_v) in enumerate(bounds_results):
    print(f"{idx:<12} | {min_v:<12.4f} | {max_v:<12.4f}")
print("="*40)


--- Building Gurobi Model for Full Network ---
Set parameter TimeLimit to value 30
Creating 28 input variables...
  > Adding Linear Layer 0: 28 inputs -> 16 outputs
  > Adding ReLU constraints...
  > Adding Linear Layer 1: 16 inputs -> 10 outputs

--- Starting Optimization for 10 Output Neurons ---
Processing Output Node 0...
Gurobi Optimizer version 12.0.1 build v12.0.1rc0 (mac64[arm] - Darwin 24.6.0 24G90)

CPU model: Apple M2 Pro
Thread count: 10 physical cores, 10 logical processors, using up to 10 threads

Non-default parameters:
TimeLimit  30

Optimize a model with 90 rows, 86 columns and 762 nonzeros
Model fingerprint: 0xc00a630f
Variable types: 70 continuous, 16 integer (16 binary)
Coefficient statistics:
  Matrix range     [9e-06, 1e+02]
  Objective range  [1e+00, 1e+00]
  Bounds range     [1e+00, 1e+02]
  RHS range        [2e-02, 1e+02]
Found heuristic solution: objective -0.3627079
Presolve removed 26 rows and 10 columns
Presolve time: 0.00s
Presolved: 64 rows, 76 columns, 

In [18]:
print(f"FastKAN Trainable Params: {sum(p.numel() for p in my_kan.parameters() if p.requires_grad)}")
print(f"TinyMLP Trainable Params: {sum(p.numel() for p in mlp_model.parameters() if p.requires_grad)}")

FastKAN Trainable Params: 600
TinyMLP Trainable Params: 634


# Alpha-Beta Crown Comparison